In [2]:
"""Data Generation for Acko Insurance Platform"""

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random
import uuid
import os

# Set random seed for reproducibility
np.random.seed(42)
random.seed(42)

In [3]:
# ============================================================================
# 1. GENERATE QUOTATION DATASET (Module 2)
# ============================================================================

def generate_quotation_data(n_samples=15000):
    """Generate synthetic quotation data for premium prediction"""
    
    # Vehicle makes and models - Indian market
    car_makes = {
        'Maruti': ['Swift', 'Baleno', 'Alto', 'Dzire', 'Brezza', 'Ertiga', 'Vitara Brezza'],
        'Hyundai': ['i20', 'Creta', 'Verna', 'Tucson', 'Alcazar', 'Grand i10'],
        'Honda': ['City', 'Amaze', 'Civic', 'WR-V', 'Jazz', 'Elevate'],
        'Toyota': ['Innova', 'Fortuner', 'Camry', 'Glanza', 'Urban Cruiser'],
        'Tata': ['Tiago', 'Altroz', 'Harrier', 'Safari', 'Nexon', 'Punch'],
        'Mahindra': ['Thar', 'Scorpio', 'XUV700', 'Bolero', 'XUV300'],
        'Kia': ['Seltos', 'Sonet', 'Carens', 'EV6'],
        'MG': ['Hector', 'Astor', 'ZS EV', 'Comet'],
        'BMW': ['X1', 'X3', '3 Series', '5 Series', 'X5'],
        'Mercedes': ['A-Class', 'C-Class', 'E-Class', 'GLA', 'GLC'],
        'Audi': ['A4', 'A6', 'Q3', 'Q5', 'Q7']
    }
    
    bike_makes = {
        'Hero': ['Splendor', 'Passion Pro', 'HF Deluxe', 'Xpulse 200', 'Glamour'],
        'Honda': ['Activa', 'Shine', 'Unicorn', 'CB Hornet', 'CB Shine SP'],
        'Bajaj': ['Pulsar', 'Dominar', 'Avenger', 'CT 100', 'Platinum'],
        'TVS': ['Apache', 'Jupiter', 'Ntorq', 'Radeon', 'Raider'],
        'Royal Enfield': ['Bullet', 'Classic', 'Himalayan', 'Meteor', 'Interceptor'],
        'Yamaha': ['FZ-S', 'R15', 'MT-15', 'Ray ZR', 'Fascino']
    }
    
    # Indian cities with tier classification
    cities = {
        'tier1': ['Mumbai', 'Delhi', 'Bangalore', 'Chennai', 'Hyderabad', 'Kolkata', 'Pune', 'Ahmedabad'],
        'tier2': ['Lucknow', 'Nagpur', 'Indore', 'Patna', 'Bhopal', 'Visakhapatnam', 'Vadodara', 'Ludhiana'],
        'tier3': ['Ajmer', 'Amritsar', 'Varanasi', 'Tiruchirappalli', 'Kochi', 'Rajkot', 'Guwahati', 'Jodhpur']
    }
    
    data = []
    
    for _ in range(n_samples):
        # Choose vehicle type
        vehicle_type = random.choice(['car', 'bike'])
        
        if vehicle_type == 'car':
            make = random.choice(list(car_makes.keys()))
            model = random.choice(car_makes[make])
            manufacturing_year = random.randint(2010, 2024)
            fuel_type = random.choices(['Petrol', 'Diesel', 'Electric', 'CNG'], weights=[0.5, 0.3, 0.1, 0.1])[0]
            idv = random.randint(200000, 1500000)
        else:
            make = random.choice(list(bike_makes.keys()))
            model = random.choice(bike_makes[make])
            manufacturing_year = random.randint(2012, 2024)
            fuel_type = random.choices(['Petrol', 'Electric'], weights=[0.85, 0.15])[0]
            idv = random.randint(50000, 300000)
        
        # City with risk factor
        tier = random.choices(['tier1', 'tier2', 'tier3'], weights=[0.4, 0.35, 0.25])[0]
        city = random.choice(cities[tier])
        
        # NCB discount
        ncb_percent = random.choices([0, 20, 25, 35, 45, 50], weights=[0.3, 0.2, 0.15, 0.15, 0.1, 0.1])[0]
        
        # Calculate base premium
        vehicle_age = 2024 - manufacturing_year
        base_premium = 0
        
        if vehicle_type == 'car':
            # Base premium calculation for car
            base_premium = 8000 + (idv * 0.045)
            base_premium += vehicle_age * 500
            base_premium += random.uniform(-2000, 2000)  # Random variation
            
            if fuel_type == 'Electric':
                base_premium *= 0.88
            elif fuel_type == 'Diesel':
                base_premium *= 1.12
            elif fuel_type == 'CNG':
                base_premium *= 0.95
            
            # Make premium adjustment
            if make in ['BMW', 'Mercedes', 'Audi']:
                base_premium *= 1.3
            elif make in ['Toyota', 'Honda']:
                base_premium *= 1.05
            
            # City risk adjustment
            if tier == 'tier1':
                base_premium *= 1.15
            elif tier == 'tier3':
                base_premium *= 0.85
        else:
            # Base premium calculation for bike
            base_premium = 1500 + (idv * 0.065)
            base_premium += vehicle_age * 200
            base_premium += random.uniform(-500, 1500)
            
            if fuel_type == 'Electric':
                base_premium *= 0.85
            
            # Make premium adjustment
            if make in ['Royal Enfield', 'Yamaha']:
                base_premium *= 1.15
            
            if tier == 'tier1':
                base_premium *= 1.1
            elif tier == 'tier3':
                base_premium *= 0.9
        
        # Apply NCB discount
        base_premium *= (1 - ncb_percent / 100)
        
        # Add realistic noise
        noise = np.random.normal(0, base_premium * 0.05)
        final_premium = max(base_premium + noise, 500)
        
        data.append({
            'user_id': str(uuid.uuid4()),
            'vehicle_type': vehicle_type,
            'vehicle_make': make,
            'vehicle_model': model,
            'manufacturing_year': manufacturing_year,
            'fuel_type': fuel_type,
            'city': city,
            'city_tier': tier,
            'idv': idv,
            'ncb_percent': ncb_percent,
            'vehicle_age': vehicle_age,
            'premium': round(final_premium, 2)
        })
    
    df = pd.DataFrame(data)
    return df

# Generate quotation data
print("Generating quotation data...")
quotation_df = generate_quotation_data(15000)
print(f"Generated {len(quotation_df)} quotation records")
print("\nSample data:")
print(quotation_df.head())

# Save the data
os.makedirs('../data/synthetic', exist_ok=True)
quotation_df.to_csv('../data/synthetic/quotation_data.csv', index=False)
print("\n✅ Saved to ../data/synthetic/quotation_data.csv")

Generating quotation data...
Generated 15000 quotation records

Sample data:
                                user_id vehicle_type vehicle_make  \
0  8da68ed4-5120-4467-9577-81246c98e42b          car       Maruti   
1  25867e1e-8c3a-4e3d-9f39-a7b0853e81e8          car       Toyota   
2  b1faec62-5cc5-49d3-8ae4-648dfe9fdfdc          car        Honda   
3  e7fee26d-ecd2-44b7-9831-bf2505eccb77         bike        Bajaj   
4  69eaa42e-7f53-4a56-b173-1ec59bcd3fb8         bike       Yamaha   

  vehicle_model  manufacturing_year fuel_type     city city_tier      idv  \
0        Ertiga                2014    Petrol   Nagpur     tier2   492632   
1      Fortuner                2018    Diesel     Pune     tier1  1377016   
2       Elevate                2016    Petrol  Kolkata     tier1   526064   
3      Platinum                2016    Petrol   Nagpur     tier2   241294   
4       Fascino                2017    Petrol  Chennai     tier1   234699   

   ncb_percent  vehicle_age   premium  
0    

In [4]:
# ============================================================================
# 2. GENERATE CLAIMS DATASET (Module 3)
# ============================================================================

def generate_claims_data(n_samples=8000):
    """Generate synthetic claims data for claims prediction"""
    
    damage_types = ['scratch', 'dent', 'crack', 'major_damage', 'total_loss']
    affected_parts_car = ['bumper', 'door', 'windshield', 'hood', 'fender', 'roof', 'rear_bumper', 'headlight']
    affected_parts_bike = ['fairing', 'handlebar', 'headlight', 'tail_lamp', 'mirror', 'exhaust', 'wheel']
    severity_levels = ['minor', 'moderate', 'major']
    
    data = []
    
    for _ in range(n_samples):
        vehicle_type = random.choice(['car', 'bike'])
        
        # Damage characteristics
        damage_type = random.choices(
            damage_types,
            weights=[0.25, 0.25, 0.2, 0.15, 0.15]
        )[0]
        
        severity = random.choices(
            severity_levels,
            weights=[0.35, 0.35, 0.30]
        )[0]
        
        if damage_type == 'total_loss':
            severity = 'major'
        
        if vehicle_type == 'car':
            affected_parts = random.sample(affected_parts_car, random.randint(1, 3))
            
            # Simulate claim amount based on damage
            base_amount = random.randint(2000, 45000)
            
            # Damage type multiplier
            damage_multipliers = {
                'scratch': random.uniform(0.4, 0.9),
                'dent': random.uniform(0.7, 1.4),
                'crack': random.uniform(1.2, 2.0),
                'major_damage': random.uniform(2.0, 3.8),
                'total_loss': random.uniform(4.0, 8.0)
            }
            
            amount_multiplier = damage_multipliers[damage_type]
            
            # Severity multiplier
            severity_multipliers = {
                'minor': 1.0,
                'moderate': 1.6,
                'major': 2.6
            }
            
            claim_amount = base_amount * amount_multiplier * severity_multipliers[severity]
            
            # Approval probability based on damage patterns
            if damage_type == 'scratch' and severity == 'minor':
                approval_prob = random.uniform(0.75, 0.95)
            elif damage_type == 'total_loss':
                approval_prob = random.uniform(0.2, 0.55)
            elif damage_type == 'major_damage' and severity == 'major':
                approval_prob = random.uniform(0.3, 0.6)
            else:
                approval_prob = random.uniform(0.4, 0.85)
        else:
            affected_parts = random.sample(affected_parts_bike, random.randint(1, 2))
            base_amount = random.randint(500, 18000)
            
            damage_multipliers = {
                'scratch': random.uniform(0.3, 0.8),
                'dent': random.uniform(0.6, 1.2),
                'crack': random.uniform(1.1, 1.7),
                'major_damage': random.uniform(1.8, 3.2),
                'total_loss': random.uniform(3.5, 6.5)
            }
            
            amount_multiplier = damage_multipliers[damage_type]
            
            severity_multipliers = {
                'minor': 1.0,
                'moderate': 1.4,
                'major': 2.0
            }
            
            claim_amount = base_amount * amount_multiplier * severity_multipliers[severity]
            
            if damage_type == 'scratch' and severity == 'minor':
                approval_prob = random.uniform(0.75, 0.95)
            elif damage_type == 'total_loss':
                approval_prob = random.uniform(0.2, 0.5)
            else:
                approval_prob = random.uniform(0.4, 0.85)
        
        # Add realistic noise
        claim_amount = max(claim_amount + np.random.normal(0, claim_amount * 0.08), 300)
        approval_prob = np.clip(approval_prob + np.random.normal(0, 0.04), 0.1, 0.99)
        
        # Determine status based on approval probability
        if approval_prob > 0.7:
            status = random.choices(['approved', 'approved', 'pending'], weights=[0.7, 0.2, 0.1])[0]
        elif approval_prob > 0.4:
            status = random.choices(['pending', 'review', 'approved'], weights=[0.4, 0.4, 0.2])[0]
        else:
            status = random.choices(['rejected', 'review', 'pending'], weights=[0.5, 0.3, 0.2])[0]
        
        # Generate dates
        incident_date = datetime.now() - timedelta(days=random.randint(0, 180))
        created_at = incident_date + timedelta(days=random.randint(1, 10))
        
        data.append({
            'claim_id': str(uuid.uuid4()),
            'user_id': str(uuid.uuid4()),
            'vehicle_type': vehicle_type,
            'policy_number': f"POL{random.randint(100000, 999999)}",
            'incident_date': incident_date.strftime('%Y-%m-%d'),
            'damage_type': damage_type,
            'affected_parts': ', '.join(affected_parts),
            'damage_severity': severity,
            'predicted_amount': round(claim_amount, 2),
            'approval_probability': round(approval_prob, 3),
            'status': status,
            'created_at': created_at.strftime('%Y-%m-%d %H:%M:%S')
        })
    
    df = pd.DataFrame(data)
    return df

# Generate claims data
print("\nGenerating claims data...")
claims_df = generate_claims_data(8000)
print(f"Generated {len(claims_df)} claims records")
print("\nSample data:")
print(claims_df.head())

claims_df.to_csv('../data/synthetic/claims_data.csv', index=False)
print("\n✅ Saved to ../data/synthetic/claims_data.csv")


Generating claims data...
Generated 8000 claims records

Sample data:
                               claim_id                               user_id  \
0  34da1a30-9624-4ed6-a6b1-e6aa2f68b9de  fcc7a2e6-cb24-448f-88c7-b5feec747c6d   
1  77b11fda-5588-4cf4-b85d-c5b8c0f59dfc  4183f6c0-0713-437a-98e4-f6137119eb5b   
2  6177e643-cb64-4b59-a0a5-1f15210175f7  a05405da-30d8-49cf-b75d-6be5c00b884f   
3  f955dd87-1864-43f3-9d81-0b4b6454984d  d5dd53d3-ea62-4cf3-ad9c-5d983cdc1e8a   
4  3d67c045-2060-4488-9b98-6db753d02f1d  ff1cf5da-7b45-40f2-9b5b-f1777672163e   

  vehicle_type policy_number incident_date   damage_type  \
0          car     POL384115    2026-02-17  major_damage   
1          car     POL922472    2026-05-28       scratch   
2          car     POL915199    2026-01-11    total_loss   
3         bike     POL928189    2025-12-20  major_damage   
4         bike     POL965578    2026-03-16          dent   

            affected_parts damage_severity  predicted_amount  \
0             doo

In [5]:
# ============================================================================
# 3. GENERATE CHAT LOGS DATA
# ============================================================================

def generate_chat_logs(n_samples=3000):
    """Generate synthetic chat logs"""
    
    questions = [
        "Does my car insurance cover flood damage?",
        "Is zero depreciation included in my base plan?",
        "What is the waiting period for Acko health insurance?",
        "How do I file a claim for my car?",
        "What add-ons are available for bike insurance?",
        "Does bike insurance cover theft?",
        "What is the NCB discount and how does it work?",
        "How to renew my policy online?",
        "What documents are needed for a claim?",
        "Is road side assistance included in the policy?",
        "How much is the premium for Honda City?",
        "What is the claim settlement ratio of Acko?",
        "Does health insurance cover pre-existing conditions?",
        "How to cancel my policy?",
        "What is the grace period for policy renewal?",
        "Can I add my spouse to my health policy?",
        "What is the difference between comprehensive and third-party insurance?",
        "How to check my policy status?",
        "What are the exclusions in motor insurance?",
        "How to transfer my NCB to a new vehicle?"
    ]
    
    intents = ['policy_qa', 'quotation', 'claims', 'general']
    
    data = []
    for _ in range(n_samples):
        question = random.choice(questions)
        intent = random.choices(
            intents,
            weights=[0.4, 0.25, 0.2, 0.15]
        )[0]
        
        # Simple mock answer
        answer = f"Based on your policy documents, here's what I found about '{question}'. Please refer to your policy booklet for complete details."
        
        created_at = datetime.now() - timedelta(days=random.randint(0, 30))
        
        data.append({
            'log_id': str(uuid.uuid4()),
            'user_id': str(uuid.uuid4()),
            'intent': intent,
            'question': question,
            'answer': answer,
            'retrieved_source': f"acko_policy_page_{random.randint(1, 50)}.pdf",
            'created_at': created_at.strftime('%Y-%m-%d %H:%M:%S')
        })
    
    df = pd.DataFrame(data)
    return df

print("\nGenerating chat logs...")
chat_df = generate_chat_logs(3000)
chat_df.to_csv('../data/synthetic/chat_logs.csv', index=False)
print("✅ Saved to ../data/synthetic/chat_logs.csv")


Generating chat logs...
✅ Saved to ../data/synthetic/chat_logs.csv


In [6]:
# ============================================================================
# 4. GENERATE USERS DATA
# ============================================================================

def generate_users_data(n_samples=800):
    """Generate synthetic users"""
    
    first_names = ['Raj', 'Priya', 'Amit', 'Sneha', 'Vikram', 'Ananya', 'Rahul', 'Meera',
                   'Arjun', 'Kavya', 'Suresh', 'Deepa', 'Manish', 'Pooja', 'Kiran',
                   'Neha', 'Ravi', 'Swati', 'Ankit', 'Divya']
    
    last_names = ['Sharma', 'Patel', 'Singh', 'Reddy', 'Kumar', 'Gupta', 'Joshi', 'Rao',
                  'Nair', 'Desai', 'Verma', 'Malhotra', 'Chopra', 'Mehta', 'Agarwal',
                  'Khanna', 'Iyer', 'Shetty', 'Goyal', 'Saxena']
    
    data = []
    for _ in range(n_samples):
        first = random.choice(first_names)
        last = random.choice(last_names)
        name = f"{first} {last}"
        email = f"{first.lower()}.{last.lower()}{random.randint(1, 100)}@example.com"
        phone = f"9{random.randint(100000000, 999999999)}"
        role = random.choices(['customer', 'manager'], weights=[0.92, 0.08])[0]
        created_at = datetime.now() - timedelta(days=random.randint(0, 365))
        
        data.append({
            'user_id': str(uuid.uuid4()),
            'name': name,
            'email': email,
            'phone': phone,
            'role': role,
            'created_at': created_at.strftime('%Y-%m-%d %H:%M:%S')
        })
    
    df = pd.DataFrame(data)
    return df

print("\nGenerating users data...")
users_df = generate_users_data(800)
users_df.to_csv('../data/synthetic/users.csv', index=False)
print("✅ Saved to ../data/synthetic/users.csv")



Generating users data...
✅ Saved to ../data/synthetic/users.csv


In [7]:
# ============================================================================
# 5. SUMMARY
# ============================================================================

print("\n" + "="*60)
print("✅ ALL SYNTHETIC DATA GENERATED SUCCESSFULLY!")
print("="*60)
print("\n📊 DATASET SUMMARY:")
print(f"  • Quotations: {len(quotation_df):,} records")
print(f"  • Claims: {len(claims_df):,} records")
print(f"  • Chat Logs: {len(chat_df):,} records")
print(f"  • Users: {len(users_df):,} records")
print("\n📁 Data saved in: ../data/synthetic/")

# Verify data
print("\n🔍 Data Verification:")
print(f"  • Quotation premium range: ₹{quotation_df['premium'].min():,.0f} - ₹{quotation_df['premium'].max():,.0f}")
print(f"  • Claim amount range: ₹{claims_df['predicted_amount'].min():,.0f} - ₹{claims_df['predicted_amount'].max():,.0f}")
print(f"  • Claim approval probability: {claims_df['approval_probability'].min():.2f} - {claims_df['approval_probability'].max():.2f}")
print(f"  • Users: {len(users_df)} customers, {len(users_df[users_df['role'] == 'manager'])} managers")



✅ ALL SYNTHETIC DATA GENERATED SUCCESSFULLY!

📊 DATASET SUMMARY:
  • Quotations: 15,000 records
  • Claims: 8,000 records
  • Chat Logs: 3,000 records
  • Users: 800 records

📁 Data saved in: ../data/synthetic/

🔍 Data Verification:
  • Quotation premium range: ₹2,548 - ₹134,920
  • Claim amount range: ₹300 - ₹1,046,444
  • Claim approval probability: 0.10 - 0.99
  • Users: 800 customers, 69 managers
